## 1. Setup & Load Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import faiss
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Dataset

In [ ]:
# Load dataset
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10335 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


## 3. Hybrid Chunking Strategy

- Metadata = 1 sentence
- Nguyên liệu theo NHÓM (5 items/group) = 2-3 sentences
- Bước nấu theo PHASE (3 steps/group) = 3-4 sentences
- Mô tả = 1 sentence

In [19]:
import ast
import pandas as pd


def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []


def hybrid_chunking(row):
    """
    Hybrid Chunking Strategy (Production-ready)

    Structure:
    - Metadata: 1 chunk
    - Ingredients: group ~5 items, avoid small tail chunks
    - Steps: group ~3 steps, avoid isolated steps
    - Description: 1 chunk
    """
    sentences = []

    # Metadata
    meta_parts = []

    if pd.notna(row.get("type_of_food")):
        meta_parts.append(str(row["type_of_food"]).strip())

    if pd.notna(row.get("title")):
        meta_parts.append(str(row["title"]).strip())

    if pd.notna(row.get("cook_time")):
        meta_parts.append(f"thời gian {row['cook_time']}")

    if pd.notna(row.get("num_of_people")):
        meta_parts.append(f"Số người ăn: {row['num_of_people']}")

    if meta_parts:
        sentences.append(". ".join(meta_parts))

    # 2. Ingredients (group by ~5, min 3)
    ingredients = parse_list_field(row.get("ingredients"))

    if ingredients:
        chunk_size = 5
        min_chunk = 3

        i = 0
        while i < len(ingredients):
            # merge small tail into previous chunk
            if len(ingredients) - i < min_chunk and sentences:
                sentences[-1] += ", " + ", ".join(ingredients[i:])
                break

            chunk = ingredients[i:i + chunk_size]
            sentences.append("Nguyên liệu: " + ", ".join(chunk))
            i += chunk_size

    # 3. Steps (group by ~3, min 2)
    steps = parse_list_field(row.get("step"))

    if steps:
        chunk_size = 3
        min_chunk = 2

        i = 0
        while i < len(steps):
            # merge isolated tail steps
            if len(steps) - i < min_chunk and sentences:
                sentences[-1] += " → " + " → ".join(steps[i:])
                break

            chunk = steps[i:i + chunk_size]
            sentences.append(" → ".join(chunk))
            i += chunk_size

    # 4. Description
    if pd.notna(row.get("description")):
        desc = str(row["description"]).strip()
        if desc:
            sentences.append(desc)

    # 5. Notes / Tips (list[str] → single semantic chunk)
    notes = parse_list_field(row.get("note"))

    if notes:
        note_text = " | ".join(notes)
        sentences.append("Lưu ý: " + note_text)

    return sentences


In [20]:
test_df = all_recipes_df[30:31]

for index, row in test_df.iterrows():
    chunks = hybrid_chunking(row)
    print(f"Recipe: {index}")
    for i, chunk in enumerate(chunks):
        print(f" Chunk {i+1}: {chunk}")
    print("\n")

Recipe: 30
 Chunk 1: Món Tết. Nộm tai heo dưa chuột giải ngán ngày Tết. thời gian 45 phút. Số người ăn: 4-5 người
 Chunk 2: Nguyên liệu: 1/2 cái tai hẹo (lợn) (250 gr), 2 quả dưa chuột, 1/2 củ cà rốt, 1/2 củ hành tây, 50 gr giá đỗ
 Chunk 3: Nguyên liệu: Chanh, tỏi, ớt, rau thơm, Gia vị: Mắm, muối, đường, hạt tiêu, Lạc rang giã dập
 Chunk 4: Bước 1: Tai heo cạo sạch lông, chà xát chanh và muối hạt khử mùi rồi rửa sạch. Đun sôi nồi nước thêm chút muối rồi cho tai vào chần để loại bỏ tạp chất, rửa sạch. → Bước 2: Cho tai heo vào nồi nước ngập, thêm 1/2 củ hành tây, gừng đập dập cho thơm, chút muối, giấm luộc khoảng 15 phút là tai chín. → Bước 3: Vớt tai heo ra ngâm ngập vào âu nước đá kèm vài lát chanh cho tai trắng giòn và thơm. Khi nguội, vớt ra thấm khô rồi thái lát mỏng. Pha nước sốt trộn nộm tỷ lệ gồm đường: nước: mắm là 1:1:1 (cho vào nồi 3 thìa canh đường: 3 thìa canh nước: 3 thìa canh nước mắm 30 độ đạm, đun sôi cho tan, để nguội). Sau đó cho thêm 1 thìa canh nước cốt chanh, 1/2 t

## 4. Generate Sentences for All Recipes

In [23]:
# Generate sentences for all recipes
all_recipes_sentences = []
for idx, row in tqdm(all_recipes_df.iterrows(), total=len(all_recipes_df), desc="Processing recipes"):
    sentences = hybrid_chunking(row)
    # hybrid_chunking on each row return list of sentences in one recipe
    all_recipes_sentences.append(sentences)

# Statistics
total_sentences = sum(len(s) for s in all_recipes_sentences)
sentences_per_recipes = [len(s) for s in all_recipes_sentences]

print(f"CHUNKING RESULTS:")
print(f"   Total recipes:        {len(all_recipes_sentences):,}")
print(f"   Total sentences:      {total_sentences:,}")
print(f"   Min sentences_per_recipes:           {min(sentences_per_recipes)}")
print(f"   Max sentences_per_recipes:           {max(sentences_per_recipes)}")
print(f"   Median sentences_per_recipes:        {np.median(sentences_per_recipes):.1f}")

Processing recipes: 100%|██████████| 10335/10335 [00:02<00:00, 4219.31it/s]

CHUNKING RESULTS:
   Total recipes:        10,335
   Total sentences:      68,784
   Min sentences_per_recipes:           2
   Max sentences_per_recipes:           22
   Median sentences_per_recipes:        7.0


In [24]:
all_recipes_sentences[:2]

[['Món Tết. Cách muối dưa hành truyền thống. thời gian 45 phút. Số người ăn: 8-10 người',
  'Nguyên liệu: 1 kg hành củ tươi, Tro bếp hoặc nước vo gọa, Muối hạt, đường, Cà rốt trang trí (tùy chọn), Lọ sạch',
  "Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, màu sắc tươi đều (tím nhạt hoặc trắng). Tránh mua hành ấn vào mềm, chảy nước hoặc mốc là đã hỏng. Chỉ nên lựa củ vừa phải, không nên to quá. → Bước 2: Ngâm khử mùi hăng của hành: Theo kinh nghiệm dân gian và trong sách ''Thế vị tân biên'' xuất bản năm 1925 đề cập để khử hăng, giúp hành giòn và trắng nên ngâm với nước tro bếp 2 - 3 ngày. Có nhà dùng nước vo gạo ngâm cũng có tác dụng tương tự. → Bước 3: Nhặt rễ, ngâm nước muối: Dùng dao nhỏ sắc, cắt gần sát rễ (không cắt hết), bóc lớp vỏ già ngoài. Sau đó, ngâm hành vào nước muối loãng ngâm khoảng 30 phút. Việc này giúp khử hành bớt hăng và khử khuẩn để khi ngâm hành không bị nổi váng, úng nhớt. Nếu muốn tăng thêm màu sắc bắt mắt, tỉa thêm ch

In [32]:
print(len(all_recipes_sentences))
print(type(all_recipes_sentences[1][1]))

10335
<class 'str'>


all_recipes_sentences = [[], [],...]

## 5. Load Vietnamese SBERT Model

In [13]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


## 6. Encode All Sentences → Multi-vector Embeddings

In [42]:
def encode_multi_vector_dishes(all_recipes_sentences, model, batch_size=64):
    """
    Encode all sentences into embeddings
    
    Returns:
        dish_embeddings_list: List of numpy arrays, mỗi món = list of vectors
        all_embeddings: Flattened embeddings cho FAISS index
        sentence_counts: Số lượng câu mỗi món cho mapping
    """
    # Flatten all sentences
    all_sentences = []
    sentence_counts = []
    
    for recipe in all_recipes_sentences:
        all_sentences.extend(recipe)
        # extend khác với append, trong khi append vẫn giữ list để thêm vào thì extend sẽ phá vỡ list để thêm vào từng phần tử
        sentence_counts.append(len(recipe))
    
    print(f"Encoding {len(all_sentences):,} sentences...")
    
    # Encode all at once
    all_flat_embeddings = model.encode(
        all_sentences,
        show_progress_bar=True,
        batch_size=batch_size
    )
    
    # Split back to per-dish
    # recipes_embeddings_list có dạng (num_dishes, num_vectors_per_dish, embedding_dim)
    recipes_embeddings_list = []
    start_idx = 0
    for count in sentence_counts:
        end_idx = start_idx + count
        recipe_embeddings = all_flat_embeddings[start_idx:end_idx]
        recipes_embeddings_list.append(recipe_embeddings)
        start_idx = end_idx
    
    print(f"Encoding completed!")
    print(f"   {len(recipes_embeddings_list):,} recipes encoded")
    print(f"   {sentence_counts[0]}-{max(sentence_counts)} vectors per recipe")
    
    return recipes_embeddings_list, all_flat_embeddings, sentence_counts

In [43]:
# Encode all dishes
dish_embeddings, all_flat_embeddings, sentence_counts = encode_multi_vector_dishes(
    all_recipes_sentences, 
    model,
    batch_size=64
)

Encoding 68,784 sentences...


Batches:   1%|▏         | 16/1075 [05:33<6:07:44, 20.84s/it]


KeyboardInterrupt: 

In [ ]:
print(f"EMBEDDING RESULTS:")
print(f"   Total recipes:        {len(dish_embeddings):,}")
print(f"   Total vectors:        {len(all_flat_embeddings):,}")
print(f"   Vector dimension:     {all_flat_embeddings.shape}")
print(f"   dish_embeddings shape:     {dish_embeddings[2].shape}")
print(f"   Example recipe 0:     {len(dish_embeddings[0])} vectors")

## 7. Build FAISS Index

Sử dụng **IndexFlatIP** (Inner Product) vì embeddings đã được normalize

In [ ]:
# Normalize embeddings for cosine similarity
print("Normalizing embeddings...")
faiss.normalize_L2(all_flat_embeddings)

# Build FAISS index
print("Building FAISS index...")
dimension = all_flat_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = Cosine similarity when normalized
index.add(all_flat_embeddings)

print(f"FAISS index built successfully!")
print(f"   Index size: {index.ntotal:,} vectors")
print(f"   Dimension:  {dimension}")

## 8. Create Mapping: Vector Index → Recipe Info

In [ ]:
# Create mapping: flat_index → (recipe_idx, sentence_idx)
vector_to_recipe_mapping = []
for recipe_idx, count in enumerate(sentence_counts):
    for sentence_idx in range(count):
        vector_to_recipe_mapping.append({
            'recipe_idx': recipe_idx,
            'sentence_idx': sentence_idx,
            'sentence_text': all_recipes_sentences[recipe_idx][sentence_idx]
        })

In [ ]:
#Example
print(f"Example mapping entry 0:")
print(f"   Recipe index: {vector_to_recipe_mapping[0]['recipe_idx']}")
print(f"   Sentence index: {vector_to_recipe_mapping[0]['sentence_idx']}")
print(f"   Text: {vector_to_recipe_mapping[0]['sentence_text'][:100]}...")

## 9. Save to Disk

In [ ]:
# Save FAISS index
faiss.write_index(index, "RARec_data/food_recipes_hybrid.index")
print("✅ Saved FAISS index: food_recipes_hybrid.index")

# Save mapping
with open("RARec_data/vector_to_recipe_mapping.pkl", "wb") as f:
    pickle.dump(vector_to_recipe_mapping, f)
print("✅ Saved mapping: vector_to_recipe_mapping.pkl")

# Save dish embeddings 
with open("RARec_data/dish_embeddings_list.pkl", "wb") as f:
    pickle.dump(dish_embeddings, f)
print("✅ Saved dish embeddings: dish_embeddings_list.pkl")

# Save recipe metadata
all_recipes_sentences.to_csv("RARec_data/recipes_metadata.csv", index=False, encoding='utf-8-sig')
print("✅ Saved metadata: recipes_metadata.csv")

# Save config
config = {
    'total_recipes': len(all_recipes_sentences),
    'total_vectors': len(all_flat_embeddings),
    'dimension': dimension,
    'model_name': 'keepitreal/vietnamese-sbert',
    'chunking_strategy': 'hybrid'
}
with open("RARec_data/embedding_config.pkl", "wb") as f:
    pickle.dump(config, f)
print("✅ Saved config: embedding_config.pkl")

## 10. Late Fusion Search Function

**Late Fusion Strategy:**
1. Tính similarity của query với **TẤT CẢ** các câu trong mỗi món
2. Lấy **trung bình** (average) similarity của tất cả câu → điểm số của món
3. Rank tất cả món theo điểm trung bình → Top K

In [ ]:
def search_recipes_late_fusion(query, model, dish_embeddings_list, df, top_k=10, verbose=True):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)
    
    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank
    
    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        dish_embeddings_list: List of embeddings per dish (from encode function)
        df: Recipe metadata dataframe
        top_k: Number of results to return
        verbose: Print detailed results
    
    Returns:
        DataFrame with top_k recipes and average similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize
    
    # 2. Calculate average similarity for EACH recipe
    print(f"Calculating similarities for {len(dish_embeddings_list)} recipes...")
    recipe_scores = []
    
    for recipe_idx, dish_embeds in enumerate(dish_embeddings_list):
        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)
        
        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()
        
        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)
        
        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'num_sentences': len(similarities),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities))
        })
    
    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]
    
    # 4. Create results dataframe
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = df.iloc[recipe_idx]
        
        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'description': recipe['description']
        })
    
    results_df = pd.DataFrame(results)
    
    # 5. Print results
    if verbose:
        print(f"\n🔍 Query: '{query}'")
        print(f"📊 Top {len(results_df)} Results:")
        print("="*100)
        for idx, row in results_df.iterrows():
            print(f"\n{idx+1}. [AVG: {row['avg_similarity']:.4f} | MAX: {row['max_similarity']:.4f}] {row['title']}")
            print(f"   Loại: {row['type_of_food']} | Thời gian: {row['cook_time']}")
            print(f"   Số câu đã đánh giá: {row['num_sentences']}")
            if pd.notna(row['description']):
                print(f"   Mô tả: {row['description'][:150]}...")
    
    return results_df


print("✅ Late Fusion search function defined!")

## 11. Test Late Fusion Search

In [ ]:
# Test Late Fusion
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
    "Món Tết truyền thống của miền Bắc",
    "Canh chua cá cho bữa trưa",
    "Món chay thanh đạm dễ làm"
]

# Run Late Fusion tests
for query in test_queries:
    print("\n" + "="*100)
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        dish_embeddings_list=dish_embeddings,
        df=df,
        top_k=10,
        verbose=True
    )
    print("="*100)

## 12. Interactive Search

In [ ]:
# Interactive search với Late Fusion
print("🔍 INTERACTIVE SEARCH MODE (LATE FUSION)")
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("\n>>> Query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("👋 Goodbye!")
        break
    
    if not query:
        continue
    
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        dish_embeddings_list=dish_embeddings,
        df=df,
        top_k=10,
        verbose=True
    )